# Notebook 08 — Pandas Fundamentals

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 08 of 20  
**Prerequisites:** Notebook 07  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Gene expression data from GEO, TCGA, or any published dataset arrives as a
tab-separated or CSV file with metadata: gene IDs, sample IDs, conditions,
batch labels, clinical annotations. NumPy arrays have no notion of row/column names;
Pandas DataFrames do. Almost every bioinformatics pipeline starts and ends with Pandas.

---
## Step 2 — Intuition

A DataFrame is a 2-D table where:
- The **index** labels rows (often gene IDs or sample IDs)
- The **columns** label columns (often sample names or feature names)
- Each column is a **Series** — a 1-D array with an index

The most common bug: treating a DataFrame index as a column or vice versa.

---
## Step 3 — Biological Background

**GEO expression matrices** typically look like this:

```
Gene_ID   GSM001   GSM002   GSM003   Organism   Gene_Symbol
---
207       5.12     4.98     6.11     Homo sap.  JAK1
208       12.3     11.8     13.2     Homo sap.  MATK
```

Mixed data: numeric expression values alongside string metadata. DataFrames handle
this cleanly; NumPy cannot (all elements must share a dtype).

**Key metadata columns to always check:** tissue, condition, batch, library_type,
sequencing_depth. Always verify these before any analysis.

---
## Step 4 — Mathematical Explanation

**GroupBy aggregation:**

For a set of groups $\mathcal{G} = \{g_1, g_2, \ldots\}$ and a statistic $f$,
`df.groupby(key)[col].agg(f)` computes $f(\{x_i : \text{group}(i) = g_j\})$ for
each group $g_j$. This is the split-apply-combine paradigm.

The Pandas implementation is O(n) for most aggregation functions (hashed grouping),
not O(n²).

---
## Step 5 — Computational Explanation

| Operation | Pandas API | Gotcha |
|-----------|-----------|--------|
| Select column | `df['col']` or `df.col` | `.col` fails if name has spaces |
| Select rows | `df.loc[label]`, `df.iloc[idx]` | loc = label, iloc = integer |
| Boolean filter | `df[df['col'] > 5]` | Creates a copy (usually) |
| GroupBy | `df.groupby('cond')['expr'].mean()` | NaN propagation by default |
| Pivot table | `df.pivot_table(...)` | Wide ↔ long transformation |
| Merge | `pd.merge(left, right, on='key')` | Like SQL JOIN |
| Missing values | `df.isna()`, `df.dropna()`, `df.fillna()` | Check before any analysis |

---
## Step 6 — Python Implementation

In [ ]:
import pandas as pd
import numpy as np
print(f"Pandas {pd.__version__}")

rng = np.random.default_rng(42)

In [ ]:
# Cell 6.1 — Building a synthetic expression DataFrame
genes = ["BRCA1", "TP53", "MYC", "EGFR", "KRAS", "PTEN", "RB1", "CDH1"]
samples = ["tumour_1", "tumour_2", "tumour_3", "normal_1", "normal_2", "normal_3"]

expr_data = rng.exponential(scale=5.0, size=(len(genes), len(samples)))

df = pd.DataFrame(expr_data, index=genes, columns=samples)
df.index.name = "gene_id"

print(df.round(2))
print(f"\nShape: {df.shape}  dtypes: {df.dtypes.unique()}")

In [ ]:
# Cell 6.2 — Indexing: loc vs iloc
print("Single gene (label):", df.loc["BRCA1"].round(2).to_dict())
print("First gene (int):  ", df.iloc[0].round(2).to_dict())
print()
print("Gene subset:")
print(df.loc[["TP53", "MYC"], ["tumour_1", "normal_1"]].round(2))
print()
print("Boolean filter — high expression in tumour_1:")
print(df[df["tumour_1"] > 5][["tumour_1", "normal_1"]].round(2))

In [ ]:
# Cell 6.3 — Adding metadata and GroupBy
# Condition metadata
meta = pd.DataFrame({
    "sample": samples,
    "condition": ["tumour"]*3 + ["normal"]*3,
    "batch": ["B1", "B1", "B2", "B1", "B2", "B2"],
}).set_index("sample")

print("Metadata:")
print(meta)

# Transpose df so samples are rows for groupby
df_T = df.T  # (samples, genes)
df_T = df_T.join(meta)

# Mean expression per condition
mean_by_cond = df_T.groupby("condition")[genes].mean()
print("\nMean expression per condition:")
print(mean_by_cond.round(2))

In [ ]:
# Cell 6.4 — Missing values
# Introduce artificial missing values
df_missing = df.copy()
df_missing.iloc[2, 1] = np.nan
df_missing.iloc[5, 3] = np.nan

print("Missing value summary:")
print(df_missing.isna().sum())
print(f"\nAny missing: {df_missing.isna().any().any()}")

# Strategy 1: drop genes with any missing
df_dropped = df_missing.dropna()
print(f"\nAfter dropna: {df_dropped.shape[0]} genes remain")

# Strategy 2: fill with gene mean
df_filled = df_missing.T.fillna(df_missing.mean(axis=1)).T
print(f"After fillna(gene_mean): all NaN gone = {not df_filled.isna().any().any()}")

In [ ]:
# Cell 6.5 — Melt (wide → long) for plotting
df_long = df.reset_index().melt(id_vars="gene_id", var_name="sample", value_name="expression")
df_long = df_long.merge(meta.reset_index(), on="sample")
print(f"Long format shape: {df_long.shape}")
print(df_long.head())

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Boxplot using long-format data
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=df_long, x="gene_id", y="expression", hue="condition",
            palette={"tumour": "tomato", "normal": "steelblue"}, ax=ax)
ax.set_title("Expression distribution per gene by condition")
ax.set_xlabel("Gene")
ax.set_ylabel("Expression")
plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Load the expression DataFrame from a CSV file (write a helper `load_expression_csv(path)`
   that reads the file, sets the gene_id index, and validates that all values are ≥ 0).
2. Write a `sample_qc(df, meta)` function that returns a DataFrame with one row per sample
   showing: total counts, median expression, fraction of genes with 0 counts.
3. Using `df_long`, compute the fold-change (tumour/normal mean) per gene.
4. What is the difference between `df.loc[mask]` and `df[mask]`? Are there cases where
   they differ?

---
## Quiz — Active Recall

1. What is the difference between `df.loc` and `df.iloc`?
2. `df.groupby('condition')['gene'].mean()` — what type does this return?
3. Why does `df.iloc[0]` sometimes return a different column than `df.loc[0]`?
4. What does `df.melt(id_vars='gene_id')` do to the shape of the DataFrame?
5. Why should you check for missing values before any groupby operation?

---
## Reflection

**Date completed:** ____________________

> *[Could you go from a raw CSV to a grouped mean per condition from memory? Which indexing gotcha tripped you up?]*

---
*Next: `09_data_cleaning_biological.ipynb`*